In [3]:
# !pip install ipywidgets ipykernel --break-system-packages

In [ ]:
# !pip install -U huggingface_hub --break-system-packages

In [ ]:
# !pip install --upgrade transformers --break-system-packages

In [ ]:
# !pip install "git+https://github.com/intel/auto-round.git@refs/pull/1656/head" --break-system-packages

In [3]:
# !hf download google/gemma-4-26B-A4B-it --local-dir ./local_model

In [4]:
# !hf download google/gemma-4-E4B-it --local-dir ./local_model_4b

In [3]:
import os
import torch
from auto_round import AutoRound
from huggingface_hub import HfApi, create_repo, notebook_login, get_token
from transformers import AutoModelForImageTextToText, AutoProcessor

In [2]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [4]:
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA Version: {torch.version.cuda}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

# -----------------------------
# FlashAttention Check
# -----------------------------
def check_flash_attention():
    try:
        import flash_attn
        from flash_attn.flash_attn_interface import flash_attn_func

        print("FlashAttention: INSTALLED ✅")

        # Optional deeper check (GPU compatibility)
        if torch.cuda.is_available():
            major, minor = torch.cuda.get_device_capability(0)
            print(f"GPU Compute Capability: {major}.{minor}")

            if major >= 8:  # Ampere (A100, RTX 30xx, etc.)
                print("FlashAttention: GPU SUPPORTED ✅")
            else:
                print("FlashAttention: GPU may NOT be fully supported ⚠️")

        return True

    except ImportError:
        print("FlashAttention: NOT INSTALLED ❌")
        return False

    except Exception as e:
        print("FlashAttention: ERROR ⚠️")
        print("Reason:", str(e))
        return False


check_flash_attention()

PyTorch Version: 2.10.0+cu126
CUDA Available: True
CUDA Version: 12.6
GPU Name: NVIDIA A40
VRAM: 44.4 GB
FlashAttention: INSTALLED ✅
GPU Compute Capability: 8.6
FlashAttention: GPU SUPPORTED ✅


True

In [4]:
MODEL_ID = "google/gemma-4-E4B-it"
HF_USER = "Vishva007"
OUTPUT_BASE_DIR = "./AutoRound"

In [5]:
model_dir = f"./local_model_4b"
model_dir_26B = f"./local_model"

In [6]:
notebook_login()

In [7]:
model = AutoModelForImageTextToText.from_pretrained(
    model_dir, 
    dtype=torch.bfloat16, 
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_dir)

tokenizer = processor.tokenizer


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

In [8]:
model

Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (language_model): Gemma4TextModel(
      (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 2560, padding_idx=0)
      (layers): ModuleList(
        (0-4): 5 x Gemma4TextDecoderLayer(
          (self_attn): Gemma4TextAttention(
            (q_norm): Gemma4RMSNorm()
            (k_norm): Gemma4RMSNorm()
            (v_norm): Gemma4RMSNorm()
            (k_proj): Linear(in_features=2560, out_features=512, bias=False)
            (q_proj): Linear(in_features=2560, out_features=2048, bias=False)
            (v_proj): Linear(in_features=2560, out_features=512, bias=False)
            (o_proj): Linear(in_features=2048, out_features=2560, bias=False)
          )
          (mlp): Gemma4TextMLP(
            (gate_proj): Linear(in_features=2560, out_features=10240, bias=False)
            (up_proj): Linear(in_features=2560, out_features=10240, bias=False)
            (down_proj): Linear(in_features=10240, out_features=2560, bias=Fals

In [12]:
model_26B = AutoModelForImageTextToText.from_pretrained(
    model_dir, 
    dtype=torch.bfloat16, 
    device_map="auto"
)
processor_26B = AutoProcessor.from_pretrained(model_dir)

tokenizer_26B = processor_26B.tokenizer

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

In [13]:
model_26B

Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (language_model): Gemma4TextModel(
      (embed_tokens): Gemma4TextScaledWordEmbedding(262144, 2560, padding_idx=0)
      (layers): ModuleList(
        (0-4): 5 x Gemma4TextDecoderLayer(
          (self_attn): Gemma4TextAttention(
            (q_norm): Gemma4RMSNorm()
            (k_norm): Gemma4RMSNorm()
            (v_norm): Gemma4RMSNorm()
            (k_proj): Linear(in_features=2560, out_features=512, bias=False)
            (q_proj): Linear(in_features=2560, out_features=2048, bias=False)
            (v_proj): Linear(in_features=2560, out_features=512, bias=False)
            (o_proj): Linear(in_features=2048, out_features=2560, bias=False)
          )
          (mlp): Gemma4TextMLP(
            (gate_proj): Linear(in_features=2560, out_features=10240, bias=False)
            (up_proj): Linear(in_features=2560, out_features=10240, bias=False)
            (down_proj): Linear(in_features=10240, out_features=2560, bias=Fals

In [ ]:
del model_26B

In [18]:
del processor_26B, tokenizer_26B

In [18]:
# %%
import re

quant_block_list = []

for name, module in model.named_modules():
    # Include: language model transformer blocks
    if re.fullmatch(r'model\.language_model\.layers\.\d+', name):
        quant_block_list.append([name])
    # Include: vision encoder transformer blocks
    elif re.fullmatch(r'model\.vision_tower\.encoder\.layers\.\d+', name):
        quant_block_list.append([name])
    # audio_tower.layers.N is intentionally excluded

print(f"[Info] Total blocks to quantize: {len(quant_block_list)}")
for b in quant_block_list:
    print(f"  {b[0]}")

[Info] Total blocks to quantize: 58
  model.language_model.layers.0
  model.language_model.layers.1
  model.language_model.layers.2
  model.language_model.layers.3
  model.language_model.layers.4
  model.language_model.layers.5
  model.language_model.layers.6
  model.language_model.layers.7
  model.language_model.layers.8
  model.language_model.layers.9
  model.language_model.layers.10
  model.language_model.layers.11
  model.language_model.layers.12
  model.language_model.layers.13
  model.language_model.layers.14
  model.language_model.layers.15
  model.language_model.layers.16
  model.language_model.layers.17
  model.language_model.layers.18
  model.language_model.layers.19
  model.language_model.layers.20
  model.language_model.layers.21
  model.language_model.layers.22
  model.language_model.layers.23
  model.language_model.layers.24
  model.language_model.layers.25
  model.language_model.layers.26
  model.language_model.layers.27
  model.language_model.layers.28
  model.language_

In [19]:
# %%
# Also add layer_config to skip small non-standard layers that aren't divisible by group_size
# (per_layer_input_gate: 256 out_features, per_layer_projection: 256 in_features)
layer_config = {}
for name, module in model.named_modules():
    # Skip entire audio tower at linear level too
    if "audio_tower" in name:
        layer_config[name] = {"data_type": "bfloat16"}
    # Skip the small per-layer gating projections (out_features=256, not divisible cleanly)
    if "per_layer_input_gate" in name or "per_layer_projection" in name:
        layer_config[name] = {"data_type": "bfloat16"}
    # Skip vision tower clippable linear wrappers (they're already tiny and crucial for accuracy)
    if "embed_vision" in name or "embed_audio" in name:
        layer_config[name] = {"data_type": "bfloat16"}

In [21]:
layer_config

{'model.language_model.layers.0.per_layer_input_gate': {'data_type': 'bfloat16'},
 'model.language_model.layers.0.per_layer_projection': {'data_type': 'bfloat16'},
 'model.language_model.layers.1.per_layer_input_gate': {'data_type': 'bfloat16'},
 'model.language_model.layers.1.per_layer_projection': {'data_type': 'bfloat16'},
 'model.language_model.layers.2.per_layer_input_gate': {'data_type': 'bfloat16'},
 'model.language_model.layers.2.per_layer_projection': {'data_type': 'bfloat16'},
 'model.language_model.layers.3.per_layer_input_gate': {'data_type': 'bfloat16'},
 'model.language_model.layers.3.per_layer_projection': {'data_type': 'bfloat16'},
 'model.language_model.layers.4.per_layer_input_gate': {'data_type': 'bfloat16'},
 'model.language_model.layers.4.per_layer_projection': {'data_type': 'bfloat16'},
 'model.language_model.layers.5.per_layer_input_gate': {'data_type': 'bfloat16'},
 'model.language_model.layers.5.per_layer_projection': {'data_type': 'bfloat16'},
 'model.language

In [20]:
TUNING_CONFIG = {
    "group_size": 128,
    "sym": True,
    "iters": 0,  # High accuracy (Production grade)
    "nsamples": 32,  # More calibration data
    "batch_size": 2,  # Faster on 48GB VRAM
    "seqlen": 2048,
    "low_gpu_mem_usage": False,  # Keep on GPU for speed
    "enable_torch_compile": True,  # JIT acceleration
    "quant_nontext_module": False,  # Keep Vision Tower in FP16 (Crucial for VLM accuracy)
    "layer_config": layer_config 
}

In [22]:
def push_to_hub(local_dir, repo_name, token):
    """Creates repo and uploads folder to Hugging Face."""
    full_repo_id = f"{HF_USER}/{repo_name}"
    print(f"\n[Hub] Pushing {local_dir} to {full_repo_id}...")

    try:
        api = HfApi()
        create_repo(
            full_repo_id, repo_type="model", exist_ok=True, private=False, token=token
        )

        api.upload_folder(
            folder_path=local_dir, repo_id=full_repo_id, repo_type="model", token=token
        )
        print(f"[Hub] ✅ Successfully uploaded: https://huggingface.co/{full_repo_id}")
    except Exception as e:
        print(f"[Hub] ❌ Error uploading: {e}")

In [23]:
ar = AutoRound(
    model=model,
    tokenizer=tokenizer,
    processor=processor,
    scheme="W4A16",
    **TUNING_CONFIG,
)

2026-04-07 02:47:13 INFO autoround.py L178: using MLLM mode for multimodal model.
2026-04-07 02:47:13 INFO base.py L473: `enable_opt_rtn` is turned on, set `--disable_opt_rtn` for higher speed at the cost of accuracy.
2026-04-07 02:47:13 INFO base.py L517: using torch.bfloat16 for quantization tuning


In [24]:
# SINGLE CALL to save all 3 formats to the same output directory
# The files will exist side-by-side or merged in this folder.
ar.quantize_and_save(
    OUTPUT_BASE_DIR, format="auto_round", inplace=True
)

IndexError: tuple index out of range

In [ ]:
base_name = MODEL_ID.split("/")[-1]
hf_token = get_token()
print(base_name)

In [ ]:
path_autoround = os.path.join(OUTPUT_BASE_DIR, "auto-round-auto-gptq")
path_gptq = os.path.join(OUTPUT_BASE_DIR, "auto-gptq")
path_awq = os.path.join(OUTPUT_BASE_DIR, "auto-awq")

In [ ]:
if hf_token:
    # 1. AutoRound Repo
    # Verify path exists before uploading
    if os.path.exists(path_autoround):
        push_to_hub(path_autoround, f"{base_name}-W4A16-AutoRound", hf_token)
    else:
        print(f"⚠️ Could not find AutoRound output at {path_autoround}")

    # 2. GPTQ Repo
    if os.path.exists(path_gptq):
        push_to_hub(path_gptq, f"{base_name}-W4A16-AutoRound-GPTQ", hf_token)
    else:
        print(f"⚠️ Could not find GPTQ output at {path_gptq}")

    # 3. AWQ Repo
    if os.path.exists(path_awq):
        push_to_hub(path_awq, f"{base_name}-W4A16-AutoRound-AWQ", hf_token)
    else:
        print(f"⚠️ Could not find AWQ output at {path_awq}")

In [ ]:
push_to_hub("./AutoRound", f"{base_name}-W4A16-AutoRound", hf_token)

In [6]:
from transformers import AutoProcessor, AutoModelForCausalLM

MODEL_ID = "Intel/gemma-4-26B-A4B-it-int4-AutoRound/"
# Load model
processor = AutoProcessor.from_pretrained(model_dir_26B)
model = AutoModelForCausalLM.from_pretrained(
    model_dir_26B,
    dtype="auto",
    device_map="auto"
)


# Prompt - add image before text
messages = [
    {
        "role": "user", "content": [
            {"type": "image", "url": "https://raw.githubusercontent.com/google-gemma/cookbook/refs/heads/main/Demos/sample-data/GoldenGate.png"},
            {"type": "text", "text": "What is shown in this image?"}
        ]
    }
]

# Process input
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

# Generate output
outputs = model.generate(**inputs, max_new_tokens=512)
response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

# Parse output
print(processor.parse_response(response))


Loading weights:   0%|          | 0/1013 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


{'role': 'assistant', 'content': "This image shows the Golden Gate Bridge, an iconic suspension bridge in San Francisco, California. The view is taken from the shoreline near Fort Point, showing the bridge's massive red towers and suspension cables stretching across the entrance to the San Francisco Bay. In the foreground, you can see a rocky coastline, a large brick building (part of the Presidio/Fort Point area), and the blue waters of the bay."}
